# HW1 — Classical Image Processing (Colab starter)
智慧醫療影像與訊號處理 · W3

依題目卷 Part B / Part C 填完 TODO。**Runtime → Run all** 要能無錯執行。
影像用 scikit-image 內建醫學影像（`data.brain()` 腦 MRI / `shepp_logan_phantom()` CT phantom），不必上傳檔案。


In [ ]:
# === Setup: run me first ===
# scikit-image is preinstalled on Colab; uncomment if you hit a version issue:
# !pip install -U scikit-image
import numpy as np
import matplotlib.pyplot as plt
from skimage import data, exposure, filters, feature, morphology, util, measure, color
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim

def load_image():
    """Return one medical image slice as float in [0, 1]."""
    try:
        vol = data.brain()                      # brain MRI (downloads on first call)
        im = vol[vol.shape[0] // 2].astype(float)
        return im / im.max()
    except Exception:
        return util.img_as_float(data.shepp_logan_phantom())

img = load_image()
print("loaded:", img.shape, img.dtype, round(float(img.min()), 3), round(float(img.max()), 3))


## B1 — 載入與檢視 Load & inspect

In [ ]:
# B1
img = load_image()
plt.figure(figsize=(4, 4)); plt.imshow(img, cmap="gray"); plt.axis("off"); plt.show()
# TODO: print shape, dtype, min, max
# print(...)


## B2 — 直方圖等化 Histogram equalization

In [ ]:
# B2
eq = exposure.equalize_hist(img)
# TODO: 2x2 subplot -> [original image | its histogram] on top, [equalized | its histogram] below
# fig, ax = plt.subplots(2, 2, figsize=(9, 7))
# ax[0,0].imshow(img, cmap="gray"); ...
# ax[0,1].hist(img.ravel(), bins=64); ...


## B3 — 去噪比較 Denoising (report PSNR & SSIM)

In [ ]:
# B3
noisy   = util.random_noise(img, mode="gaussian", var=0.02)
mean_f  = filters.rank.mean(util.img_as_ubyte(noisy), morphology.disk(2)) / 255.0
gauss_f = filters.gaussian(noisy, sigma=1.5)
med_f   = filters.median(noisy, morphology.disk(2))
# TODO: for each of {noisy, mean_f, gauss_f, med_f} compute PSNR & SSIM vs clean img,
#       print a table, and display the four images. Which is best? Why?
# print("gauss:", psnr(img, gauss_f, data_range=1.0), ssim(img, gauss_f, data_range=1.0))


## B4 — 邊緣偵測 Edge detection

In [ ]:
# B4
sob = filters.sobel(img)
# TODO: apply feature.canny with >= 3 different sigma values; display; comment on how edge count changes
# for s in [1, 2, 3]:
#     edges = feature.canny(img, sigma=s)
#     ...


## B5 — 分割 Segmentation

In [ ]:
# B5
mask = img > filters.threshold_otsu(img)
# TODO: morphology.remove_small_objects(mask, ...) -> measure.label -> color.label2rgb overlay; display


## B6 — 頻域濾波 Frequency-domain filtering

In [ ]:
# B6
F   = np.fft.fftshift(np.fft.fft2(img))
mag = np.log(1 + np.abs(F))
# TODO: build a low-pass mask (keep center) and a high-pass mask (keep outer),
#       reconstruct each with np.fft.ifft2(np.fft.ifftshift(F * mask)), display spectrum + both results


## Part C — 迷你專題 Pipeline
自選一張醫學影像，串 增強 → 去噪 → (邊緣 或 分割)。量化評估 (PSNR/SSIM)，並在報告寫 200–400 字討論。

In [ ]:
# Part C — your pipeline
img = load_image()
# TODO: chain enhance -> denoise -> (edge or segment); evaluate; visualize


## 打包工具庫（必做，HW2 起要 import）
把上面 Part B 寫過的函式整理成三個 `.py` 存到自己的 Google Drive。

In [ ]:
# 1) 掛載 Drive：
# from google.colab import drive; drive.mount('/content/drive')
# !mkdir -p /content/drive/MyDrive/medimg_toolkit
#
# 2) 用 %%writefile 打包（獨立 cell 執行；函式內容換成你 Part B 寫好的版本）：
# %%writefile /content/drive/MyDrive/medimg_toolkit/mypreproc.py
# def normalize(img, mode='minmax'): ...
# def denoise(img, method='median'): ...
# def augment(img, flip=True, rot=True): ...
#
# 同法建立 mymetrics.py（psnr, ssim）與 myviz.py（show_pair, overlay）
#
# 3) 之後每份作業開頭這樣用：
# import sys; sys.path.append('/content/drive/MyDrive/medimg_toolkit')
# from mypreproc import normalize

## Part D — 逆向工程（繳交介面固定）

助教會用**你沒看過的保留測例**呼叫下面兩個函式，所以：

- 函式名稱、參數順序、回傳格式 **不可更動**，回傳 `np.float32`。
- 不要在函式裡讀檔或用全域變數存參數，推出來的常數請寫死在函式內。
- 函式要能重複呼叫，不依賴前面 cell 的執行順序。
- 報告要寫推理過程：至少比較兩個假說、附支持的圖或數字、說明哪裡不可辨識。

門檻與檔案格式見 `reverse/manifest.json`。下面的自評格只檢查公開資料，通過不代表保留測例一定過。

In [ ]:
# === Part D 資料載入 ===
# Colab：把課程發的 reverse/ 整個資料夾放進自己的 Drive 再掛載，或用 files.upload() 上傳。
# from google.colab import drive; drive.mount('/content/drive')
import os, json
import numpy as np

REVERSE = '/content/drive/MyDrive/HW1/reverse'   # ← 改成你的路徑
man = json.load(open(os.path.join(REVERSE, 'manifest.json'), encoding='utf-8'))
def L(name): return np.load(os.path.join(REVERSE, name))
def LL(names): return [np.load(os.path.join(REVERSE, x)) for x in names]

D1_in, D1_tg = LL(man['D1']['files']['inputs']), LL(man['D1']['files']['targets'])
D2_in, D2_tg = LL(man['D2']['files']['inputs']), LL(man['D2']['files']['targets'])
print('D1:', len(D1_in), D1_in[0].shape, D1_in[0].dtype, '| 門檻', man['D1']['threshold'])
print('D2:', len(D2_in), D2_in[0].shape, D2_in[0].dtype, '| 門檻', man['D2']['threshold'])

In [ ]:
# === D1 灰階與模糊的先後之謎 ===
from skimage.metrics import structural_similarity as ssim_fn

def report(rows):
    for i, (txt, ok) in enumerate(rows):
        print(f'[{i}] {txt}  ' + ('PASS' if ok else 'FAIL'))
    print('== ALL PASS ==' if all(ok for _, ok in rows) else '== 尚未達標 ==')

def reproduce_D1(inputs):
    # inputs: list of (128,128) float32 in [0,1] -> list of (128,128) float32
    # 只能用線性窗位 clip((x-lo)/(hi-lo),0,1)、gamma、gaussian_filter 三種操作
    from scipy.ndimage import gaussian_filter
    outs = []
    for x in inputs:
        x = np.asarray(x, dtype=np.float64)
        # TODO: 依你推出的「順序 + 參數」重建處理鏈
        y = x
        outs.append(y.astype(np.float32))
    return outs

# --- 自評：SSIM >= 0.98 且 MAE <= 0.005，逐張都要過 ---
pred = reproduce_D1(D1_in)
rows = []
for p, t in zip(pred, D1_tg):
    s = ssim_fn(t, p, data_range=1.0); m = float(np.abs(t - p).mean())
    rows.append((f'SSIM={s:.5f} MAE={m:.6f}', s >= 0.98 and m <= 0.005))
report(rows)

In [ ]:
# === D2 頻譜上的手術 ===
def reproduce_D2(inputs):
    # inputs: list of (128,128) float32（含條紋，值可能超出 [0,1]）-> list of (128,128) float32
    # 只能用 2D DFT 域的 Gaussian 低通 + 最多兩對共軛 notch，取 ifft2 的實部
    outs = []
    for x in inputs:
        X = np.fft.fftshift(np.fft.fft2(np.asarray(x, dtype=np.float64)))
        # TODO: 建出 H = 低通 x notch，套用後反變換
        H = np.ones_like(X, dtype=np.float64)
        y = np.real(np.fft.ifft2(np.fft.ifftshift(X * H)))
        outs.append(y.astype(np.float32))
    return outs

# --- 自評：MAE <= 0.0008 且 PSNR >= 53 dB，逐張都要過 ---
def psnr1(t, p):
    mse = float(np.mean((t - p) ** 2))
    return 99.0 if mse == 0 else 10 * np.log10(1.0 / mse)

pred2 = reproduce_D2(D2_in)
rows = []
for p, t in zip(pred2, D2_tg):
    m = float(np.abs(t - p).mean()); q = psnr1(t, p)
    rows.append((f'MAE={m:.2e} PSNR={q:.2f}dB', m <= 0.0008 and q >= 53.0))
report(rows)

# 找 notch 位置的起手式：看殘差頻譜能量集中在哪幾個 bin
# R = np.fft.fftshift(np.fft.fft2(D2_tg[0] - pred2[0]))
# plt.imshow(np.log1p(np.abs(R))); plt.colorbar()